# argmax-accuracy-eval — ex1: top-1 classification accuracy from logits

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `argmax-accuracy-eval`. Running the final beacon cell reports progress against the `Eval: argmax accuracy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Eval: argmax accuracy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`argmax-accuracy-eval`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "argmax-accuracy-eval"
DD_SUBTOPIC = "Eval: argmax accuracy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `(logits.argmax(dim=-1) == labels).float().mean()` — quick refresher

Top-1 classification accuracy in three idioms:

```
preds   = logits.argmax(dim=-1)             # (B,) predicted class
correct = (preds == labels)                 # (B,) boolean
acc     = correct.float().mean()            # scalar in [0, 1]
```

**Why `argmax(dim=-1)`.** `logits` is `(B, C)`. We want the index of the largest logit ALONG THE CLASS AXIS for each example. `dim=-1` is the class axis regardless of whether there are extra leading dims (e.g. `(B, T, C)` for token-level outputs).

**Why `.float().mean()` and not `.sum() / len(labels)`.** Boolean tensors can't be `.mean()`-ed directly. Casting to float gives `1.0 / 0.0` per example, and `.mean()` then handles partial-last-batch sizes correctly when accumulated across batches with a weighted average.

**Logits vs probabilities — argmax is the same.** Softmax is monotonic, so `argmax(logits) == argmax(softmax(logits))`. You can skip the softmax for accuracy; the predicted class is identical.

### Exercise 1 — top-1 classification accuracy from logits

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the `(logits.argmax(dim=-1) == labels).float().mean()` accuracy pattern, including the dim=-1 axis choice and the boolean-to-float cast.
> Keywords: accuracy, argmax, eval-metric, classification
> ```

**KCs targeted:** `argmax-along-class-dim-minus-1`, `boolean-tensor-float-mean-accuracy`

Implement `ex1_top1_accuracy(logits, labels)`. The standard classification eval metric.

1. Compute `preds = logits.argmax(dim=-1)` — shape `(B,)`.
2. Compute `correct = (preds == labels)` — shape `(B,)`, dtype bool.
3. Return `correct.float().mean()` — a scalar tensor in `[0, 1]`.

Inputs:
- `logits`: `(B, C)` float tensor.
- `labels`: `(B,)` int tensor with class indices in `[0, C)`.

Output: scalar accuracy tensor.

**Critical:** use `dim=-1` (the class axis) not `dim=0` or `dim=1` explicitly — `dim=-1` correctly handles `(B, C)` and `(B, T, C)` (token-level) shapes alike. The test verifies you got this right by passing both shapes.

In [ ]:
def ex1_top1_accuracy(logits, labels):
    preds = logits.argmax(dim=-1)
    return (preds == labels).float().mean()


<details><summary>Solution</summary>

```python
def ex1_top1_accuracy(logits, labels):
    preds = logits.argmax(dim=-1)
    return (preds == labels).float().mean()
```

**Why bool → float → mean.** `t.tensor([True, False]).mean()` raises `RuntimeError: Can only calculate the mean of floating types`. The fix is `(preds == labels).float()` before `.mean()`. A common almost-right form is `correct.sum() / correct.numel()` — works but is less idiomatic and slightly slower (two ops vs one).

**Why argmax is enough — no softmax needed.** Softmax is MONOTONIC: if `logits[a] > logits[b]` then `softmax(logits)[a] > softmax(logits)[b]`. So `argmax(logits) == argmax(softmax(logits))`. Doing the softmax first is harmless but wasteful.

**Where this lives in real training code.** Inside `validate()` you accumulate accuracy over the val loader, weighted by batch size (just like loss). For top-K accuracy you replace `argmax(dim=-1)` with `topk(k, dim=-1).indices` and check whether `labels` is in that set.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()